# A001 Beginner AI Project
## Is the pump reading Normal or Abnormal?

This is intentionally a **very simple first machine-learning project**.

We will use only **one input variable**:

**Vibration → Decision Tree → Normal / Abnormal**

### Business question
Can we look at the vibration reading of asset **A-001** and predict whether the reading is abnormal?

### Important
This notebook is **not predicting an actual pump failure**.  
The A-001 data does not contain real historical failure examples.

Instead, we use the existing `anomaly_flag` in the sensor data:

- `0` = Normal
- `1` = Abnormal

This lets beginners understand the complete ML flow before moving to forecasting or failure prediction.


## Step 1 — Import the libraries

We only need a few basic Python libraries.


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix


## Step 2 — Connect to the A-001 database

The database is stored in the Databricks Volume you created.


In [ ]:
DB_PATH = "/Volumes/workspace/v2_nuclear_enterprise_360/v2_nuclear_enterprise_360/nuclear_enterprise_360_v2_2_clean.db"

conn = sqlite3.connect(DB_PATH)

query = '''
SELECT
    reading_timestamp,
    asset_id,
    vibration_mm_s,
    anomaly_flag
FROM sensor_readings
WHERE asset_id = 'A-001'
ORDER BY reading_timestamp
'''

df = pd.read_sql_query(query, conn)
conn.close()

print("Rows loaded:", len(df))
df.head(10)


## Step 3 — Understand the data

For this first project we only care about two columns:

- `vibration_mm_s` → the input to the model
- `anomaly_flag` → what we want the model to predict


In [ ]:
print("Normal vs abnormal readings:")
print(df["anomaly_flag"].value_counts())

df[["vibration_mm_s", "anomaly_flag"]].describe()


## Step 4 — Look at vibration visually

This is just to help us see how vibration changes over time.


In [ ]:
df["reading_timestamp"] = pd.to_datetime(df["reading_timestamp"])

plt.figure(figsize=(12, 4))
plt.plot(df["reading_timestamp"], df["vibration_mm_s"])
plt.xlabel("Time")
plt.ylabel("Vibration (mm/s)")
plt.title("A-001 Vibration Readings")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Step 5 — Tell the model what is X and what is Y

In machine learning:

- **X = input**
- **Y = answer we want to predict**

Here:

- X = vibration
- Y = normal or abnormal


In [ ]:
X = df[["vibration_mm_s"]]
y = df["anomaly_flag"]

print("X shape:", X.shape)
print("Y shape:", y.shape)


## Step 6 — Split the data into training and testing

The model learns from the **training data**.

Then we check it using **testing data** that it did not train on.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## Step 7 — Train a very simple Decision Tree

We deliberately use a tree with only **one decision**.

That makes it easy to explain:

> If vibration is below a learned value → Normal  
> If vibration is above a learned value → Abnormal


In [ ]:
model = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained.")


## Step 8 — Let the model predict

Now the model predicts `0` or `1` for the test data.


In [ ]:
predictions = model.predict(X_test)

results = X_test.copy()
results["Actual"] = y_test.values
results["Predicted"] = predictions

results.head(15)


## Step 9 — Check how well it worked

We will keep the evaluation simple.

### Confusion Matrix
It tells us:

- How many normal readings were correctly identified
- How many abnormal readings were correctly identified


In [ ]:
accuracy = accuracy_score(y_test, predictions)
cm = confusion_matrix(y_test, predictions)

print("Accuracy:", round(accuracy * 100, 2), "%")
print()
print("Confusion Matrix:")
print(cm)

print()
print("Matrix meaning:")
print("[[Correct Normal, Wrongly called Abnormal],")
print(" [Missed Abnormal, Correct Abnormal]]")


## Step 10 — See the rule the model learned

Because we limited the tree to one decision, we can directly see the vibration split learned from the training data.


In [ ]:
threshold = model.tree_.threshold[0]

print(f"The model learned a vibration split of approximately {threshold:.2f} mm/s")
print()
print(f"If vibration <= {threshold:.2f} mm/s  → predict NORMAL")
print(f"If vibration >  {threshold:.2f} mm/s  → predict ABNORMAL")


## Step 11 — Visualise the simple decision tree


In [ ]:
plt.figure(figsize=(10, 5))
plot_tree(
    model,
    feature_names=["vibration_mm_s"],
    class_names=["Normal", "Abnormal"],
    filled=False,
    rounded=True
)
plt.title("A-001 Beginner Decision Tree")
plt.show()


## Step 12 — Try new vibration readings

Now pretend these are new readings arriving from A-001.

The model will answer:

**Normal or Abnormal?**


In [ ]:
new_readings = pd.DataFrame({
    "vibration_mm_s": [1.2, 3.0, 4.8, 5.2, 5.5]
})

new_readings["prediction"] = model.predict(new_readings)
new_readings["meaning"] = new_readings["prediction"].map({
    0: "NORMAL",
    1: "ABNORMAL"
})

new_readings


# What did we learn?

We completed a full AI / machine-learning project with just one sensor:

**A-001 vibration → model → Normal / Abnormal**

The learner has now seen:

1. Read data from a database
2. Choose an input
3. Choose a target
4. Split training and testing data
5. Train a model
6. Make predictions
7. Evaluate the result
8. Use the model on new readings

## Business interpretation

An **abnormal prediction does not mean the pump has failed**.

It means:

> “This vibration reading looks like the readings that were historically marked as abnormal.”

That prediction can trigger the next step:

**Prediction → inspect A-001 → review work orders / condition reports / procedures → human decision**

This is a good foundation before teaching time-series forecasting, multivariate models, or true failure prediction.
